# HVAC preset gallery

This notebook connects the HVAC presets in the Excel tool to proposed section drawings.
We inspect the projects used in the other how-tos, preserve the original preset names,
and distinguish available symbols from drawings that still need implementation.

The gallery is a design review. It does not apply presets, change scenario inputs,
or introduce automatic preset-to-layout inference.

In [ ]:
from pathlib import Path
import json
import math
import hashlib
from html import escape
from xml.etree import ElementTree as ET
import pandas as pd
from IPython.display import display, HTML, SVG
from pexl import Project
from pexl.spatial import section_layout_from_scenario, render_svg
from _hvac_gallery import mapping, gallery_svg

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
catalog_path = ROOT / "docs/development/hvac_preset_catalog.json"
catalog = json.loads(catalog_path.read_text(encoding="utf-8"))
proposals = mapping(catalog)

def show_svg(svg):
    ET.fromstring(svg)  # Validate the output before displaying it.
    display(HTML('<div style="overflow:auto">' + svg + '</div>'))


## 1. Sources and observed projects

`howto_excel` and `howto_excel_updated` load the v1_12 project backup;
`howto_schema_transition` uses v1_13. `howto_sankey` uses `Project.example()`.
The electricity and grid-load how-tos refer to an external Aspern pickle;
we do not load that external file. The v1_13 backup includes Aspern variants.

The preset snapshot comes from the heating, cooling, hot-water and ventilation
sheets of the development workbook. The generic `PRESETS` sheet mainly contains
weather and assessment entries, so it is not the complete HVAC catalogue.
The snapshot records the workbook hash, exact sheet/cell ranges, names and cached
values. It can be inspected without opening or recalculating Excel.

In [ ]:
sources = {
    "howto_excel / updated": ROOT / "data/exports/ka_project_backup_v1_12.xlsx",
    "howto_schema_transition": ROOT / "data/exports/ka_project_backup_v1_13.xlsx",
    "howto_sankey / Project.example": ROOT / "data/exports/development_examples.xlsx",
}
projects = {name: Project.from_excel(path) for name, path in sources.items() if path.exists()}
display(pd.DataFrame([
    {"How-to": name, "File": path.name, "Available": path.exists(),
     "Scenarios": len(projects[name].scenarios) if name in projects else 0,
     "Project names": ", ".join(projects[name].project_names()) if name in projects else "Missing locally"}
    for name, path in sources.items()
]))
source_workbook = ROOT / catalog["source_workbook"]
print("Preset source:", catalog["source_workbook"])
print("Catalogue entries:", len(proposals))
print("Workbook matches snapshot:", source_workbook.exists() and
      hashlib.sha256(source_workbook.read_bytes()).hexdigest() == catalog["sha256"])


In [ ]:
preset_fields = [
    "preset_recorded_heating_system", "preset_recorded_heating_emission_system",
    "preset_recorded_cooling_system", "preset_recorded_cooling_modes",
    "preset_recorded_ventilation_system", "preset_recorded_dhw_system",
]
def present(value):
    return value is not None and not (isinstance(value, float) and math.isnan(value)) and str(value).strip() != ""

observations = []
for source, project in projects.items():
    for field in preset_fields:
        found = [getattr(s.v, field, None) for s in project.scenarios]
        observations.append({"Source": source, "Variable": field,
            "Recorded / total": f"{sum(present(v) for v in found)} / {len(found)}",
            "Recorded names": "; ".join(sorted({str(v) for v in found if present(v)})) or "Not recorded"})
display(pd.DataFrame(observations))


Most exported generation and emission preset names are missing. A scenario title
containing `WP`, or a coefficient of performance greater than one, may suggest a
heat pump but does not establish its source or its room-side emitter.

For reliable mapping, use this order:

1. Explicit per-building and per-usage layout choices.
2. Recorded preset name, checked against populated component inputs.
3. Positive capacity, carrier and usage-share values for the particular component.
4. An unresolved component where evidence is insufficient.

A carrier name in an inactive slot is not an installed generator. Fractional
usage shares should be labelled or partitioned explicitly, not rounded to a whole
building. Plant location and central/decentral distribution remain layout choices.

## 2. Mapping catalogue

Names below are read from the workbook, including spelling variants such as
`Mono-/Mulit-Splitgerät, Einzelraum`. The internal name and dropdown label can differ.
Blank preset cells are omitted from the snapshot; they are not assumed to be zero.

**Available** means the existing renderer has the relevant symbol. **Partial**
means it covers only part of the component or topology. **Proposed** means a new
symbol is needed. Even available symbols are not automatically inferred here.

In [ ]:
mapping_table = pd.DataFrame([
    {"Category": r["category"], "Preset": r["name"], "Dropdown label": r["display_name"],
     "Proposed representation": r["visual"], "Renderer support": r["support"],
     "Source cells": r["source"], "Review note": r["caution"]}
    for r in proposals
])
display(mapping_table)


## 3. Room-side emission

The catalogue distinguishes concrete activation, underfloor pipes, high-temperature
radiators, cooling panels and low-temperature heat-pump radiators. A cooling panel
belongs below the ceiling, while TABS belongs inside the concrete. A low-temperature
radiator does not automatically imply a fan coil.

The preset capacities are model inputs, not dimensions for the drawing. Their presence
alone does not enable heating or cooling in every usage.

In [ ]:
boards = {}
boards["emission"] = gallery_svg([r for r in proposals if r["category"] == "Emission"], "Room-side emission")
show_svg(boards["emission"])

## 4. Heating generation

Separate the generator, its source, and its room-side emitter. Show a heat-pump
cabinet with a ground loop, groundwater connection or outdoor fan as appropriate.
`Erdwärme` does not prove vertical boreholes: use the borehole drawing only when
`borehole_count` or an explicit layout specification supports it.

The drawings below are proposed compositions. The current renderer already covers
boilers, district heating, wells and boreholes; complete heat-pump assemblies and
outdoor split units are additions for review.

In [ ]:
boards["heating"] = gallery_svg([r for r in proposals if r["category"] == "Heating"], "Heating generation and sources")
show_svg(boards["heating"])

## 5. Cooling generation and modes

The catalogue contains overlapping names and several hybrid systems. Keep central
chillers, district cooling and local split units distinct. `Freecooling` needs a
source decision; it should not silently become window ventilation. The combined
`Wärmepumpe mit Grundwasser oder Erdwärme` explicitly leaves the source unresolved.

Some entries contain values that deserve review: for example, `Geothermie Freecooling`
also carries electric cooling capacity. We preserve these values rather than correcting
them in the drawing layer. The two boards retain every cooling preset name.

In [ ]:
cooling_rows = [r for r in proposals if r["category"] == "Cooling"]
for page, start in enumerate(range(0, len(cooling_rows), 9), 1):
    key = f"cooling_{page}"
    boards[key] = gallery_svg(cooling_rows[start:start+9], f"Cooling catalogue - page {page}")
    show_svg(boards[key])

## 6. Ventilation

Exhaust-only and balanced ventilation need different flow arrows. An exhaust system
with heat recovery is not automatically a balanced supply/extract system. This is a
limitation of mapping both to the current generic `heat_recovery` drawing.

Use `vent_share_mechanical_<usage>` to determine which usages have mechanical
ventilation, and `vent_heat_recovery_winter_<usage>` for the recovery label.
Night-ventilation presets describe window operation and belong to the window/air-path
layer, not to a new mechanical plant symbol.

In [ ]:
boards["ventilation"] = gallery_svg([r for r in proposals if r["category"] == "Ventilation"], "Ventilation topology")
show_svg(boards["ventilation"])

## 7. Domestic hot water

Central and decentral systems should occupy different places in the section.
Show local heaters at draw-off points; central systems use a plant-room generator,
a storage tank and distribution/return pipes. Use `DHW_storage_1_liter` and
`DHW_storage_2_liter` for storage evidence. An energy carrier alone does not determine
centrality or equipment count.

A shared space-heating/DHW generator should be drawn once, with separate branches.
The current renderer does not yet distinguish these DHW arrangements.

In [ ]:
boards["hot_water"] = gallery_svg([r for r in proposals if r["category"] == "Hot water"], "Domestic hot water - local and central")
show_svg(boards["hot_water"])

## 8. Inspect the exact inputs behind a card

Select a name to see the original preset values and the schema labels. Names that
are absent from the current schema are shown explicitly so migration issues remain
visible. This inspection does not apply the preset to a scenario.

In [ ]:
selected_preset = next(r for r in proposals if r["kind"] == "ground_hp")
reference = next(iter(projects.values())).scenarios[0]
metadata = {meta.attr_name: meta for meta, _ in reference.select().items()}
display(pd.DataFrame([
    {"Variable": key, "Value": value,
     "Schema label": metadata[key].label_de if key in metadata else "Not in current schema - review alias",
     "Source": selected_preset["source"]}
    for key, value in selected_preset["values"].items()
]))

## 9. Existing projects: evidence and section examples

These sections use actual scenario areas and envelope inputs from the v1_13 how-to
project. Forsthausgasse uses the previously requested two-building 2:1 length layout;
the Gründerzeit example uses one schematic building. These are explicit layout
overrides, not recovered building geometry. Building-specific plant locations are
unknown, so confirmed systems are shown as shared defaults.

The conservative mapping below draws active gas/district-heat slots and per-usage
ventilation. It leaves room-side emitters unspecified. Heat-pump source, DHW topology
and cooling emitters remain review items rather than guessed equipment.

In [ ]:
def number(scenario, name):
    try:
        value = float(getattr(scenario.v, name, None))
        return value if math.isfinite(value) else None
    except (ValueError, TypeError):
        return None

usage_keys = ["residential", "office", "schoolprim", "schoolsec", "retailfood", "retailother", "other"]
def supported_overlay(scenario):
    plant, notes = [], []
    for slot in (2, 4):
        power = number(scenario, f"QHmax_{slot}th_m2")
        carrier = str(getattr(scenario.v, f"heat_th{slot}_carrier_type", ""))
        if power is not None and power > 0:
            if "Fernw" in carrier: plant.append("district_heating")
            elif carrier == "Erdgas": plant.append("gas_boiler")
            elif "Biomasse" in carrier: plant.append("biomass_boiler")
            else: notes.append(f"Thermal slot {slot}: unresolved carrier {carrier}")
    if (number(scenario, "QHmax_1el_m2") or 0) > 0:
        notes.append("Electric heat generator active; source and emitter need review.")
    usages = {}
    for usage in usage_keys:
        if (number(scenario, "NFA_"+usage) or 0) <= 0: continue
        share = number(scenario, "vent_share_mechanical_"+usage)
        recovery = number(scenario, "vent_heat_recovery_winter_"+usage)
        if share == 1:
            usages[usage] = {"ventilation": "heat_recovery" if (recovery or 0)>0 else "mechanical"}
            notes.append(f"{usage}: generic mechanical symbol; exhaust/balanced topology unverified.")
        elif share == 0:
            notes.append(f"{usage}: no mechanical ventilation; window-operation details omitted.")
        else:
            notes.append(f"{usage}: partial/unknown mechanical share {share}; omitted from whole-room overlay.")
    return {"usages": usages, "plant": sorted(set(plant)), "roof": ["pv"]}, notes

project = projects.get("howto_schema_transition")
real_sections = {}
if project is not None:
    selected = [s for s in project.scenarios if s.column_name in {
        "Forsthausgasse | BT2 KON FW", "Forsthausgasse | BT2 ÖKO WP FLEX 2,6xPV",
        "Gründerzeit Beispiel | Saniert mit GWWP"}]
    for scenario in selected:
        display(HTML("<h3>"+escape(scenario.column_name)+"</h3>"))
        overlay, notes = supported_overlay(scenario)
        usages = [u for u in usage_keys if (number(scenario, "NFA_"+u) or 0)>0]
        if "residential" in usages: usages.remove("residential"); usages.append("residential")
        weights = [2, 1] if scenario.column_name.startswith("Forsthausgasse") else [1]
        floors = max(1, math.ceil(number(scenario, "storey_count_avg") or 4))
        layout = {"systems": overlay, "buildings": [
            {"name": f"Building {i+1}", "relative_length": weight, "depth_m": 15,
             "max_floors": floors, "usages": usages}
            for i, weight in enumerate(weights)]}
        evidence_names = ["preset_recorded_heating_system", "preset_recorded_heating_emission_system",
                          "QHmax_1el_m2", "heat_cop_1el", "QHmax_2th_m2", "heat_th2_carrier_type",
                          "borehole_count", "DHW_carriertype_1", "DHW_storage_1_liter"]
        display(pd.DataFrame({"Variable": evidence_names,
                              "Scenario value": [getattr(scenario.v, key, None) for key in evidence_names]}))
        section = section_layout_from_scenario(scenario, layout=layout)
        real_sections[scenario.column_name] = render_svg(section)
        show_svg(real_sections[scenario.column_name])
        display(HTML("<ul>"+"".join("<li>"+escape(note)+"</li>" for note in notes)+"</ul>"))
else:
    print("The v1_13 project is unavailable locally; the preset gallery remains usable.")

## 10. Review decisions and next steps

The main additions suggested by the catalogue are heat-pump assemblies and outdoor
units, central chillers, district cooling, suspended cooling panels, separate
exhaust/balanced ventilation, and local/central DHW equipment.

Before automatic mapping, decide:

- How unresolved heat-pump sources should appear: an explicit unknown-source cabinet,
  or a prompt for ground, groundwater or air.
- Where central equipment belongs when the scenario contains no building assignment.
- Whether fractional usage shares should subdivide rooms or remain labelled overlays.

Keep ambiguous presets explicit. Do not infer TABS from heat-pump efficiency or a
source type, and do not convert exhaust-only heat recovery to balanced ventilation.
The figures in this notebook are review proposals, not new public layout keywords.

## Technical notes

The workbook is read only during the catalogue extraction; no macros or formulas
are executed. The JSON snapshot includes exact cell ranges and a workbook hash.
The notebook uses the existing `Project`, `Scenario`, `scenario.v` and schema metadata.
The drawing helper is notebook-local and does not alter `pexl` inference.

The following optional export cell writes only to a directory you choose.

In [ ]:
EXPORT_DIR = None  # Example: Path("hvac_gallery_exports")
if EXPORT_DIR is not None:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    for name, svg in boards.items():
        (EXPORT_DIR / f"{name}.svg").write_text(svg, encoding="utf-8")
    mapping_table.to_csv(EXPORT_DIR / "mapping.csv", index=False)

assert len(proposals) == len(catalog["presets"]) == 40
assert all(ET.fromstring(svg).tag.endswith("svg") for svg in boards.values())
print(f"Validated {len(proposals)} mappings, {len(boards)} gallery boards and {len(real_sections)} project sections.")

## Real-project showcase

Continue with [Forsthausgasse: real-project showcase](howto_section_real_project.ipynb)
for a side-by-side comparison of two stored project variants, full-size sections
and an audit of the source inputs.